# 05 — İDA Görüntülerinde Yüzen Atık Tespiti (YOLOv8n) — AS6

> **Önemli:** Bu modül **açık veri setleriyle** yapılmıştır. Kendi çektiğimiz İDA görüntüsü yoktur ve kullanılan görüntüler **Türkiye kıyılarından değildir.** Doğru çerçeve şudur: "Türkiye kıyılarında (ör. İstanbul, İzmir) İDA ile yapılabilecek atık izleme için, açık veri setleri üzerinde bir tespit modeli geliştirip değerlendirdik. Türkiye kıyı verisiyle doğrulama gelecek çalışmadır."

**Veri seti seçimi (`config.IDA_DATASET`):**

| Değer | Veri seti | Durum |
|---|---|---|
| `"mendeley"` (şu an) | *Floating Waste and Aquatic Vegetation Image Dataset from Turbid Rivers* — Radam ve ark., Mendeley Data, v2, DOI 10.17632/j26w4m645z.2, **CC BY 4.0**. 1.577 görüntü; 2 sınıf (yüzen atık, nehir bitkisi); YOLO formatında; aynı videodan gelen kareler aynı kümede. Banjarmasin (Endonezya) nehirleri. Görüntülerin hangi platformdan (İDA, drone, el kamerası) çekildiği veri sayfasında **belirtilmemiş.** | Doğrudan indirilir (~1.76 GB, Colab geçici diskine) |
| `"flow"` | **FloW-Img** — Cheng ve ark., ICCV 2021. İnsansız su üstü aracının kamerasından çekilmiş; Pascal VOC formatında. | orca-tech.cn üzerinden **başvuruyla**. Onay gelirse zip'i Drive'a koyup değeri `"flow"` yapın |

**Nesne tespiti nedir?** Sınıflandırma "bu görüntüde ne var?" sorusunu cevaplar. Nesne tespiti ise "nerede ve kaç tane var?" sorusunu cevaplar: her nesne için bir **kutu (bounding box)** ve bir sınıf üretir.

**YOLO (You Only Look Once):** Görüntüye tek bir kez bakıp tüm kutuları aynı anda tahmin eden, hızlı bir derin öğrenme modelidir. **YOLOv8n** bu ailenin en küçük ("nano") sürümüdür. COCO veri setinde önceden eğitilmiş ağırlıklarla başlıyoruz; buna **transfer learning** denir. Model genel nesne şekillerini zaten bildiği için bizim küçük veri setimizle hızlı öğrenir.

**GPU gerekli:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**.

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) özel GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Özel repoyu okumak için Colab'ın **Gizli anahtarlar (Secrets)** bölümündeki `GITHUB_TOKEN` kullanılır. Token ekrana yazdırılmaz.

In [ ]:
!pip install -q ultralytics

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git hatası: " + (r.stderr or r.stdout).replace(token, "***"))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        raise RuntimeError("Colab'da sol menüdeki anahtar simgesinden GITHUB_TOKEN ekleyin ve "
                           "'Not defteri erişimi'ni açın (README'ye bakın).") from e
    if os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["-C", CODE_DIR, "pull", "-q"], token)
    else:
        _git(["clone", "-q", f"https://{token}@github.com/{GITHUB_REPO}.git", CODE_DIR], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import glob, hashlib, json, random, shutil, zipfile, urllib.request
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
T, F = config.TABLES_DIR, config.FIGURES_DIR
IDA_RES = "results/ida"                 # Drive: model ve eğitim çıktıları
os.makedirs(IDA_RES, exist_ok=True)
WORK = "/content/ida" if os.path.exists("/content") else os.path.abspath("ida_work")
DS = config.IDA_DATASET
random.seed(config.RANDOM_STATE); np.random.seed(config.RANDOM_STATE)
print("Veri seti:", DS, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK (T4 GPU seçin!)")
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{F}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

## 1. Veriyi indir ve YOLO formatına getir
**YOLO etiket formatı:** Her görüntü için aynı adlı bir `.txt` dosyası vardır. Her satır bir nesnedir: `sınıf_no x_merkez y_merkez genişlik yükseklik`. Koordinatlar görüntü boyutuna bölünmüş, 0–1 arası değerlerdir.

- **Mendeley:** Zaten YOLO formatında. Zip açıldıktan sonra train/valid/test klasörleri ve sınıf adları (veri setinin `data.yaml` ya da `classes.txt` dosyasından) **otomatik bulunur.** Sınıf adlarını biz uydurmuyoruz; bulunamazsa notebook durur ve klasör yapısını yazdırır.
- **FloW:** Pascal VOC (`.xml`) formatında. Aşağıdaki dönüştürücü XML kutularını (`xmin, ymin, xmax, ymax` piksel) YOLO formatına çevirir. Orijinal eğitim/test ayrımı korunur, böylece literatürle karşılaştırılabilir. Doğrulama kümesi eğitim kümesinin **son %15'lik dilimi** olarak ayrılır. Kareler video sırasıyla numaralandığı için ardışık karelerin eğitim ve doğrulamaya bölünmesi (veri sızıntısı) en aza iner.

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

def find_class_names(root):
    for p in glob.glob(f"{root}/**/*.yaml", recursive=True):
        txt = open(p, encoding="utf-8", errors="ignore").read()
        if "names" in txt:
            import yaml
            names = yaml.safe_load(txt).get("names")
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names)]
            if names:
                return list(names), p
    for pat in ("classes.txt", "obj.names", "_darknet.labels"):
        for p in glob.glob(f"{root}/**/{pat}", recursive=True):
            names = [l.strip() for l in open(p, encoding="utf-8") if l.strip()]
            if names:
                return names, p
    return None, None

def split_of(path):
    parts = [p.lower() for p in path.replace("\\", "/").split("/")]
    for s, keys in [("test", ("test",)), ("val", ("valid", "val", "validation")), ("train", ("train",))]:
        if any(k in parts for k in keys):
            return s
    return None

def print_tree(root, depth=3):
    for dirpath, dirs, files in os.walk(root):
        lvl = dirpath[len(root):].count(os.sep)
        if lvl <= depth:
            print("  " * lvl + os.path.basename(dirpath) + f"/  ({len(files)} dosya)")

def prepare_mendeley():
    raw = f"{WORK}/mendeley_raw"
    zp = f"{WORK}/mendeley.zip"
    if not os.path.exists(raw):
        os.makedirs(WORK, exist_ok=True)
        if not os.path.exists(zp):
            cands = ([config.MENDELEY_ZIP] if os.path.exists(config.MENDELEY_ZIP) else []) + \
                    sorted(glob.glob(f"{config.DATA_RAW}/*j26w4m645z*.zip") + glob.glob(f"{config.DATA_RAW}/*Floating*.zip"))
            if cands:
                config.MENDELEY_ZIP = cands[0]
                print("Bulunan zip:", cands[0])
            if not os.path.exists(config.MENDELEY_ZIP):
                raise FileNotFoundError(
                    "Mendeley zip'i Drive'da bulunamadı. Mendeley, Colab'dan doğrudan indirmeye izin vermiyor (403).\n"
                    "1) Tarayıcıda https://data.mendeley.com/datasets/j26w4m645z/2 → 'Download All'\n"
                    f"2) İnen zip'i Drive'da şu konuma koyun: {os.path.abspath(config.MENDELEY_ZIP)}\n"
                    "3) Bu hücreyi tekrar çalıştırın.")
            print("Zip Drive'dan Colab diskine kopyalanıyor (~1.76 GB)...")
            shutil.copy(config.MENDELEY_ZIP, zp)
        if sha256(zp) != config.MENDELEY_SHA256:
            print("UYARI: SHA-256 Mendeley'in yayımladığı değerle uyuşmuyor. 'Download All' farklı paketlenmiş "
                  "olabilir; içerik aşağıda kontrol edilecek (sınıf adları, küme sayıları).")
        with zipfile.ZipFile(zp) as z:
            z.extractall(raw)
        os.remove(zp)
        for inner in glob.glob(f"{raw}/**/*.zip", recursive=True):   # "Download All" zip içinde zip verebilir
            with zipfile.ZipFile(inner) as z:
                z.extractall(os.path.dirname(inner))
            os.remove(inner)
    names, src = find_class_names(raw)
    if not names:
        print_tree(raw)
        raise RuntimeError("Sınıf adları dosyası bulunamadı. Yukarıdaki klasör yapısını bize gönderin.")
    print("Sınıf adları:", names, "← kaynak:", os.path.relpath(src, raw))
    items = []
    for img in glob.glob(f"{raw}/**/*", recursive=True):
        if not img.lower().endswith(IMG_EXT):
            continue
        s = split_of(os.path.relpath(img, raw))
        lbl = os.path.splitext(img.replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}")
                               .replace("/images/", "/labels/"))[0] + ".txt"
        if not os.path.exists(lbl):                       # etiket görüntüyle aynı klasörde olabilir
            lbl = os.path.splitext(img)[0] + ".txt"
        items.append({"split": s, "image": img, "label": lbl if os.path.exists(lbl) else None})
    df = pd.DataFrame(items)
    n_none = df["split"].isna().sum()
    if n_none:
        print(f"Not: train/val/test klasörü dışında kalan {n_none} görüntü atlandı (ör. önizleme).")
    return names, df.dropna(subset=["split"])

def prepare_flow():
    zp = config.FLOW_ZIP
    if not os.path.exists(zp):
        raise FileNotFoundError(f"FloW zip'i bulunamadı: Drive'da {os.path.abspath(zp)}. README'deki adımlara bakın.")
    raw = f"{WORK}/flow_raw"
    if not os.path.exists(raw):
        with zipfile.ZipFile(zp) as z:
            z.extractall(raw)
    xmls = glob.glob(f"{raw}/**/*.xml", recursive=True)
    imgs = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f"{raw}/**/*", recursive=True)
            if p.lower().endswith(IMG_EXT)}
    sets = {}
    for s in ("train", "test", "val", "trainval"):
        for p in glob.glob(f"{raw}/**/ImageSets/Main/{s}.txt", recursive=True):
            sets[s] = {l.strip() for l in open(p) if l.strip()}
    names = sorted({o.find("name").text.strip() for x in xmls for o in ET.parse(x).getroot().iter("object")})
    print("Sınıf adları (XML'den):", names)
    lbl_dir = f"{WORK}/flow_labels"
    os.makedirs(lbl_dir, exist_ok=True)
    items = []
    for x in sorted(xmls):
        root = ET.parse(x).getroot()
        stem = os.path.splitext(root.findtext("filename") or os.path.basename(x))[0]
        if stem not in imgs:
            stem = os.path.splitext(os.path.basename(x))[0]
        if stem not in imgs:
            continue
        W, H = float(root.find("size/width").text), float(root.find("size/height").text)
        lines = []
        for o in root.iter("object"):
            b = o.find("bndbox")
            x1, y1, x2, y2 = (float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax"))
            lines.append(f"{names.index(o.find('name').text.strip())} {(x1 + x2) / 2 / W:.6f} "
                         f"{(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
        lp = f"{lbl_dir}/{stem}.txt"
        open(lp, "w").write("\n".join(lines))
        if "test" in sets:
            s = "test" if stem in sets["test"] else "train"
        else:
            s = split_of(os.path.relpath(x, raw)) or split_of(os.path.relpath(imgs[stem], raw))
        items.append({"split": s, "image": imgs[stem], "label": lp, "stem": stem})
    df = pd.DataFrame(items)
    tr = df[df["split"] == "train"].sort_values("stem")
    n_val = int(len(tr) * config.IDA_VAL_FRACTION)
    df.loc[tr.index[-n_val:], "split"] = "val"
    return names, df.drop(columns="stem")

names, items = prepare_mendeley() if DS == "mendeley" else prepare_flow()
items = items.reset_index(drop=True)
print(items["split"].value_counts())
assert set(items["split"]) >= {"train", "val", "test"}, "train/val/test kümelerinden biri bulunamadı."

In [ ]:
# Ultralytics'in beklediği düzen: WORK/yolo/{images,labels}/{train,val,test}
YOLO_DIR = f"{WORK}/yolo_{DS}"
if not os.path.exists(YOLO_DIR):
    for s in ("train", "val", "test"):
        os.makedirs(f"{YOLO_DIR}/images/{s}", exist_ok=True)
        os.makedirs(f"{YOLO_DIR}/labels/{s}", exist_ok=True)
    for i, r in items.iterrows():
        stem = f"{i:05d}_" + "".join(ch if ch.isalnum() or ch in "-_" else "_"
                                     for ch in os.path.splitext(os.path.basename(r["image"]))[0])
        ext = os.path.splitext(r["image"])[1]
        shutil.copy(r["image"], f"{YOLO_DIR}/images/{r['split']}/{stem}{ext}")
        dst = f"{YOLO_DIR}/labels/{r['split']}/{stem}.txt"
        if r["label"]:
            shutil.copy(r["label"], dst)
        else:
            open(dst, "w").close()          # nesnesiz (arka plan) görüntü
DATA_YAML = f"{YOLO_DIR}/data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\n"
            f"names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(names)))
print(open(DATA_YAML).read())

## 2. Veri seti tanıtımı (EDA)
- Küme başına görüntü ve nesne sayısı, sınıf dağılımı.
- **Kutu boyutu dağılımı:** COCO tanımına göre kutular üç gruba ayrılır. **Küçük:** alan < 32² piksel. **Orta:** 32²–96². **Büyük:** > 96². Küçük nesneler tespit için en zor gruptur. Yüzen atıklar (şişe, poşet) genelde uzaktan küçük görünür.

In [ ]:
rows = []
for s in ("train", "val", "test"):
    img_of = {os.path.splitext(f)[0]: f"{YOLO_DIR}/images/{s}/{f}" for f in os.listdir(f"{YOLO_DIR}/images/{s}")}
    for lp in glob.glob(f"{YOLO_DIR}/labels/{s}/*.txt"):
        stem = os.path.splitext(os.path.basename(lp))[0]
        ip = img_of[stem]
        W, H = Image.open(ip).size
        lines = [l.split() for l in open(lp) if l.strip()]
        if not lines:
            rows.append({"split": s, "image": stem, "W": W, "H": H, "cls": None})
        for c, x, y, w, h in [l[:5] for l in lines]:
            rows.append({"split": s, "image": stem, "W": W, "H": H, "cls": names[int(c)],
                         "w_px": float(w) * W, "h_px": float(h) * H})
boxes = pd.DataFrame(rows)
b = boxes.dropna(subset=["cls"]).copy()
b["area_px"] = b["w_px"] * b["h_px"]
b["boyut"] = pd.cut(b["area_px"], [0, 32 ** 2, 96 ** 2, np.inf], labels=["küçük", "orta", "büyük"])

eda = pd.DataFrame({
    "goruntu": boxes.groupby("split")["image"].nunique(),
    "nesnesiz_goruntu": boxes[boxes["cls"].isna()].groupby("split")["image"].nunique(),
    "nesne": b.groupby("split").size(),
}).reindex(["train", "val", "test"]).fillna(0).astype(int)
eda.loc["toplam"] = eda.sum()
cls_tab = pd.crosstab(b["split"], b["cls"]).reindex(["train", "val", "test"])
size_tab = pd.crosstab(b["cls"], b["boyut"], normalize="index").mul(100).round(1)
info = pd.DataFrame([
    ("Veri seti", DS), ("Sınıflar", ", ".join(names)),
    ("Lisans", "CC BY 4.0" if DS == "mendeley" else "orca-tech.cn başvuru koşulları (rapora yazılacak)"),
    ("Kaynak", "Radam ve ark., Mendeley Data, DOI 10.17632/j26w4m645z.2" if DS == "mendeley"
     else "Cheng ve ark., ICCV 2021, doi:10.1109/ICCV48922.2021.01077"),
    ("Türkiye verisi mi?", "HAYIR"),
    ("Görüntü boyutları", ", ".join(f"{w}×{h}" for (w, h) in boxes[["W", "H"]].drop_duplicates().head(5).values)),
], columns=["Özellik", "Değer"])
for name_, tab in [("ida_veri_ozeti", info), ("ida_kume_sayilari", eda), ("ida_sinif_dagilimi", cls_tab),
                   ("ida_kutu_boyutu", size_tab)]:
    tab.to_csv(f"{T}/{name_}.csv", encoding="utf-8-sig", index=name_ != "ida_veri_ozeti")
display(info); display(eda); display(cls_tab); display(size_tab)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
cls_tab.T.plot(kind="bar", ax=axes[0], colormap="Set2"); axes[0].set_title("Sınıf dağılımı (nesne sayısı)")
axes[0].tick_params(axis="x", rotation=0)
sns.histplot(data=b, x=np.sqrt(b["area_px"]), hue="cls", bins=50, ax=axes[1], element="step")
for v in (32, 96):
    axes[1].axvline(v, color="k", ls="--", lw=0.8)
axes[1].set_xlabel("√(kutu alanı) [piksel]"); axes[1].set_title("Kutu boyutu (kesikli: 32 ve 96 px)")
size_tab.plot(kind="barh", stacked=True, ax=axes[2], color=["#d73027", "#fee08b", "#1a9850"])
axes[2].set_xlabel("%"); axes[2].set_title("Küçük / orta / büyük nesne payı")
savefig("ida_eda")

## 3. YOLOv8n eğitimi (transfer learning)
- **Epoch:** Modelin tüm eğitim verisini bir kez görmesidir. 50 epoch kullanıyoruz.
- **imgsz = 640:** Görüntüler eğitimde 640 piksele ölçeklenir.
- Ultralytics eğitim sırasında **veri artırma (augmentation)** uygular: rastgele çevirme, renk değişimi, mozaik. Böylece model küçük veri setinde ezberlemez.
- En iyi ağırlıklar (`best.pt`) doğrulama kümesindeki mAP'e göre seçilir. Test kümesi eğitimde hiç kullanılmaz.

Eğitilmiş model Drive'da varsa yeniden eğitilmez.

In [ ]:
from ultralytics import YOLO
RUN = f"yolov8n_{DS}"
best_drive = f"{IDA_RES}/{RUN}/weights/best.pt"
if os.path.exists(best_drive):
    print("Eğitilmiş model Drive'dan yükleniyor:", best_drive)
else:
    model = YOLO(config.YOLO_MODEL)
    model.train(data=DATA_YAML, epochs=config.YOLO_EPOCHS, imgsz=config.YOLO_IMG_SIZE, seed=config.RANDOM_STATE,
                deterministic=True, project=f"{WORK}/runs", name=RUN, exist_ok=True, plots=True, verbose=False)
    shutil.copytree(f"{WORK}/runs/{RUN}", f"{IDA_RES}/{RUN}", dirs_exist_ok=True)
model = YOLO(best_drive)

### Eğitim kayıp eğrileri
- **box_loss:** Kutu konumundaki hata.
- **cls_loss:** Sınıf hatası.
- **dfl_loss:** Kutu kenarlarının hassasiyeti.

Eğitim ve doğrulama kayıpları birlikte düşüyorsa model öğreniyordur. Doğrulama kaybı yükselmeye başlarken eğitim kaybı düşmeye devam ediyorsa model **aşırı öğrenme (overfitting)** yapıyordur.

In [ ]:
hist = pd.read_csv(f"{IDA_RES}/{RUN}/results.csv")
hist.columns = [c.strip() for c in hist.columns]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, k in zip(axes[:3], ["box_loss", "cls_loss", "dfl_loss"]):
    ax.plot(hist["epoch"], hist[f"train/{k}"], label="eğitim")
    ax.plot(hist["epoch"], hist[f"val/{k}"], label="doğrulama")
    ax.set_title(k); ax.set_xlabel("epoch"); ax.legend()
axes[3].plot(hist["epoch"], hist["metrics/mAP50(B)"], label="mAP@0.5")
axes[3].plot(hist["epoch"], hist["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
axes[3].set_title("Doğrulama mAP"); axes[3].set_xlabel("epoch"); axes[3].legend()
savefig("ida_egitim_egrileri")

## 4. Test kümesinde değerlendirme
| Metrik | Anlamı |
|---|---|
| **Precision** | Modelin bulduğu kutuların yüzde kaçı gerçekten nesne? (Yanlış alarm az mı?) |
| **Recall** | Gerçek nesnelerin yüzde kaçı bulundu? (Kaçırma az mı?) |
| **F1** | Precision ve Recall'un harmonik ortalaması |
| **IoU** | Tahmin kutusu ile gerçek kutunun kesişim alanı / birleşim alanı. 0.5 üstü genelde "doğru konum" sayılır |
| **mAP@0.5** | IoU ≥ 0.5 iken, her sınıfın precision-recall eğrisi altındaki alanın (AP) sınıflar üzerinden ortalaması |
| **mAP@0.5:0.95** | IoU eşiği 0.5'ten 0.95'e kadar 0.05 adımla değiştirilerek hesaplanan mAP'lerin ortalaması. Kutu konumunun **hassasiyetini** de ölçer, daha zor bir metriktir |

Hız: Görüntü başına çıkarım süresi (ms). Gerçek zamanlı İDA uygulaması için önemlidir.

In [ ]:
val_dir = f"{WORK}/runs/{RUN}_test"
m = model.val(data=DATA_YAML, split="test", imgsz=config.YOLO_IMG_SIZE, project=f"{WORK}/runs",
              name=f"{RUN}_test", exist_ok=True, plots=True, verbose=False)
P, R = float(m.box.mp), float(m.box.mr)
rows = [{"sinif": "TÜMÜ (ortalama)", "precision": P, "recall": R, "f1": 2 * P * R / (P + R) if P + R else 0,
         "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map)}]
for i, c in enumerate(m.box.ap_class_index):
    p, r = float(m.box.p[i]), float(m.box.r[i])
    rows.append({"sinif": names[int(c)], "precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0,
                 "mAP50": float(m.box.ap50[i]), "mAP50_95": float(m.box.ap[i])})
ida = pd.DataFrame(rows).round(4)
ida["model"], ida["veri_seti"], ida["n_test_goruntu"] = "YOLOv8n", DS, int(eda.loc["test", "goruntu"])
ida["cikarim_ms_goruntu"] = round(m.speed["inference"], 2)
ida.to_csv(f"{T}/ida_metrikleri.csv", index=False, encoding="utf-8-sig")
shutil.copytree(val_dir, f"{IDA_RES}/{RUN}_test", dirs_exist_ok=True)
for fn in os.listdir(val_dir):          # PR eğrisi ve karışıklık matrisini figures/ altına da kopyala
    if fn.endswith(".png") and ("PR_curve" in fn or "confusion_matrix" in fn):
        shutil.copy(f"{val_dir}/{fn}", f"{F}/ida_{fn}")
ida

In [ ]:
pr = [f for f in os.listdir(val_dir) if "PR_curve" in f]
if pr:
    plt.figure(figsize=(7, 5)); plt.imshow(Image.open(f"{val_dir}/{pr[0]}")); plt.axis("off")
    plt.title("Precision-Recall eğrisi (test)"); plt.show()

**Neye bakmalıyız:**
- Sınıflar arasında fark var mı? Atık ile bitkinin ayrılması zor olabilir; ikisi de yüzen düzensiz şekillerdir.
- mAP@0.5 ile mAP@0.5:0.95 arasındaki fark büyükse model nesneyi **buluyor** ama kutuyu **tam oturtamıyor** demektir. Küçük nesnelerde bu beklenen bir durumdur (EDA'daki küçük nesne payına bakın).
- Çıkarım süresi T4 GPU'da görüntü başına birkaç milisaniye ise model gerçek zamanlı kullanıma uygundur. İDA üzerindeki gömülü donanımda daha yavaş olacağını not edin.

## 5. Örnek tespitler: doğru, kaçırılan, yanlış alarm
Test görüntülerinde tahminler (güven ≥ 0.25) gerçek kutularla IoU ≥ 0.5 ölçütüne göre eşleştiriliyor:
- **Yeşil:** Doğru tespit (TP)
- **Kırmızı:** Yanlış alarm (FP): model kutu çizdi ama orada gerçek nesne yok
- **Sarı:** Kaçırılan nesne (FN): gerçek nesne var ama model bulamadı

Rapor ve sunum için en çok doğru tespit içeren, en çok kaçırma içeren ve en çok yanlış alarm içeren görüntülerden toplam 9 örnek seçiliyor.

In [ ]:
import matplotlib.patches as patches

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

test_imgs = sorted(f"{YOLO_DIR}/images/test/{f}" for f in os.listdir(f"{YOLO_DIR}/images/test"))
records = []
for res in model.predict(test_imgs, conf=config.IDA_CONF, imgsz=config.YOLO_IMG_SIZE, stream=True, verbose=False):
    H, W = res.orig_shape
    stem = os.path.splitext(os.path.basename(res.path))[0]
    gt = []
    for l in open(f"{YOLO_DIR}/labels/test/{stem}.txt"):
        if l.strip():
            c, x, y, w, h = map(float, l.split()[:5])
            gt.append((int(c), [(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H]))
    preds = [(int(c), b.tolist()) for c, b in zip(res.boxes.cls.cpu(), res.boxes.xyxy.cpu())]
    matched, tp, fp = set(), [], []
    for c, pb in preds:
        best = max(((iou(pb, gb), j) for j, (gc, gb) in enumerate(gt) if gc == c and j not in matched), default=(0, None))
        if best[0] >= config.IDA_IOU_MATCH:
            matched.add(best[1]); tp.append(pb)
        else:
            fp.append(pb)
    fn = [gb for j, (gc, gb) in enumerate(gt) if j not in matched]
    records.append({"path": res.path, "tp": tp, "fp": fp, "fn": fn})

rec = pd.DataFrame({"path": [r["path"] for r in records], "n_tp": [len(r["tp"]) for r in records],
                    "n_fp": [len(r["fp"]) for r in records], "n_fn": [len(r["fn"]) for r in records]})
chosen = list(dict.fromkeys(
    list(rec.sort_values("n_tp", ascending=False).index[:3]) + list(rec.sort_values("n_fn", ascending=False).index[:3])
    + list(rec.sort_values("n_fp", ascending=False).index[:3])))[:9]
fig, axes = plt.subplots(3, 3, figsize=(18, 13))
for ax, i, title in zip(axes.ravel(), chosen, ["en çok doğru"] * 3 + ["en çok kaçırma"] * 3 + ["en çok yanlış alarm"] * 3):
    r = records[i]
    ax.imshow(Image.open(r["path"]))
    for key, color in [("tp", "lime"), ("fp", "red"), ("fn", "yellow")]:
        for x1, y1, x2, y2 in r[key]:
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, lw=1.5))
    ax.set_title(f"{title}: TP={len(r['tp'])} FP={len(r['fp'])} FN={len(r['fn'])}", fontsize=10); ax.axis("off")
fig.legend(handles=[patches.Patch(color=c, label=l) for c, l in
                    [("lime", "doğru tespit (TP)"), ("red", "yanlış alarm (FP)"), ("yellow", "kaçırılan (FN)")]],
           loc="lower center", ncol=3)
plt.subplots_adjust(bottom=0.04)
savefig("ida_ornek_tespitler")
tot = rec[["n_tp", "n_fp", "n_fn"]].sum()
print(f"conf ≥ {config.IDA_CONF}, IoU ≥ {config.IDA_IOU_MATCH}: TP={tot.n_tp}, FP={tot.n_fp}, FN={tot.n_fn}")

## 6. Literatürle karşılaştırma şablonu
Aynı veri setinde raporlanmış sonuçları siz makalelerden dolduracaksınız. Bizim satırımız otomatik ekleniyor.
- **Mendeley veri seti** 2026'da yayımlandığı için karşılaştırılabilir çalışma az olabilir. Veri setinin kendi makalesi/açıklaması varsa oradaki temel sonuçları kullanın.
- **FloW-Img** kullanılamadı (başvuru sitesi Türkiye'den açılmadı). FloW makalesindeki (Cheng ve ark., ICCV 2021) ve sonraki çalışmalardaki sonuçlar tabloya yalnızca **"farklı veri setinde raporlanan değerler"** olarak eklenir; bizim sonuçlarımızla doğrudan karşılaştırılmaz.

In [ ]:
lit_csv = f"{T}/ida_literatur_karsilastirma.csv"
ours = ida[ida["sinif"] == "TÜMÜ (ortalama)"].iloc[0]
lit = pd.DataFrame([
    {"calisma": "Bu çalışma", "veri_seti": DS, "model": "YOLOv8n", "mAP50": ours["mAP50"],
     "mAP50_95": ours["mAP50_95"], "precision": ours["precision"], "recall": ours["recall"], "kaynak": "-"},
] + [{"calisma": "[DOLDURULACAK]", "veri_seti": DS, "model": "", "mAP50": "", "mAP50_95": "",
      "precision": "", "recall": "", "kaynak": ""} for _ in range(4)])
lit.to_csv(lit_csv, index=False, encoding="utf-8-sig")
lit

## Faz 5 bitti mi?
- [ ] `results/tables/ida_metrikleri.csv`: Precision, Recall, F1, mAP@0.5, mAP@0.5:0.95 (sınıf bazında ve ortalama)
- [ ] `results/figures/ida_eda.png`, `ida_egitim_egrileri.png`, `ida_*PR_curve*.png`, `ida_ornek_tespitler.png` (9 örnek)
- [ ] README ve rapor iskeletinde "Türkiye verisi değildir" notu

## Bu fazda ne yaptık, neden yaptık
- Kendi İDA görüntümüz olmadığı için açık, lisansı belli bir veri setiyle yüzen atık tespit modeli geliştirdik. Bunun Türkiye verisi olmadığını her yerde açıkça belirttik.
- Önceden eğitilmiş YOLOv8n ile transfer learning yaptık. Küçük veri setinde sıfırdan eğitime göre çok daha hızlı ve başarılıdır.
- Veri sızıntısını önlemek için aynı videodan gelen kareleri aynı kümede tuttuk. Test kümesini eğitimde hiç kullanmadık.
- Hem tespit kalitesini (mAP, precision, recall) hem de hızı (ms/görüntü) raporladık. İDA'da gerçek zamanlı kullanım için ikisi de önemli.
- Doğru tespit, kaçırma ve yanlış alarm örneklerini görselleştirerek modelin nerede zorlandığını gösterdik (ör. küçük nesneler, yansımalar).